In [7]:
import pandas as pd

df = pd.read_csv(
    r"F:\iot\datasets\Okui22\KDDI-IoT-2019-main\KDDI-IoT-2019-main\ipfix\json\pinh19_ipfix_features_1s.csv"
)

In [8]:
print(sorted(df["device_id"].unique()))


['amazon_echo_gen2', 'au_network_camera', 'au_wireless_adapter', 'bitfinder_awair_breathe_easy', 'candy_house_sesami_wi-fi_access_point', 'google_home_gen1', 'i-o_data_qwatch', 'irobot_roomba', 'jvc_kenwood_cu-hb1', 'jvc_kenwood_hdtv_ip_camera', 'line_clova_wave', 'link_japan_eremote', 'mouse_computer_room_hub', 'nature_remo', 'panasonic_doorphone', 'philips_hue_bridge', 'planex_camera_one_shot!', 'planex_smacam_outdoor', 'planex_smacam_pantilt', 'powerelectric_wi-fi_plug', 'qrio_hub', 'sony_bravia', 'sony_network_camera', 'sony_smart_speaker', 'xiaomi_mijia_led']


In [9]:
device_to_category = {
    "amazon_echo_gen2": "smart_speaker",
    "google_home_gen1": "smart_speaker",
    "sony_smart_speaker": "smart_speaker",

    "au_network_camera": "network_camera",
    "i_o_data_qwatch": "network_camera",
    "planex_camera_one_shot!": "network_camera",
    "planex_smacam_outdoor": "network_camera",
    "planex_smacam_pantilt": "network_camera",
    "sony_network_camera": "network_camera",

    "au_wireless_adapter": "iot_gateway",
    "jvc_kenwood_cu_hb1": "iot_gateway",
    "mouse_computer_room_hub": "iot_gateway",

    "bitfinder_awair_breathe_easy": "environment_sensor",

    "candy_house_sesami_wifi_access_point": "door_lock",
    "orio_hub": "door_lock",

    "irobot_roomba": "robot_cleaner",

    "link_japan_remote": "remote_controller",
    "nature_remo": "remote_controller",

    "panasonic_doorphone": "doorphone",

    "philips_hue_bridge": "light",
    "xiaomi_mijia_led": "light",

    "powerlectric_wifi_plug": "plug",

    "sony_bravia": "tv",
}


In [28]:
device_to_category.update({
    "candy_house_sesami_wi-fi_access_point": "door_lock",
    "i-o_data_qwatch": "network_camera",
    "jvc_kenwood_cu-hb1": "iot_gateway",
    "jvc_kenwood_hdtv_ip_camera": "network_camera",
    "line_clova_wave": "smart_speaker",
    "link_japan_eremote": "remote_controller",
    "powerelectric_wi-fi_plug": "plug",
    "qrio_hub": "door_lock",
})


In [29]:
df["category"] = df["device_id"].map(device_to_category)


In [30]:
print(df["category"].isna().sum())
df[df["category"].isna()]["device_id"].unique()


0


array([], dtype=object)

In [31]:
FEATURES = ["total_bytes", "mean_bytes", "std_bytes"]

X = df[FEATURES]
y = df["category"]

print("Categories:", y.nunique())
print(y.value_counts())


Categories: 11
category
smart_speaker         2750207
network_camera        2731293
iot_gateway           1756967
tv                    1183909
doorphone              839856
light                  511186
door_lock              405961
remote_controller      332201
environment_sensor     253842
plug                   239105
robot_cleaner          124224
Name: count, dtype: int64


In [33]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import label_binarize
from sklearn.metrics import average_precision_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_score = rf.predict_proba(X_test)

y_test_bin = label_binarize(y_test, classes=rf.classes_)

aucpr = average_precision_score(
    y_test_bin,
    y_score,
    average="macro"
)

print(f"Category-level AUCPR: {aucpr:.3f}")


Category-level AUCPR: 0.659


In [36]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import average_precision_score

# 1. One-vs-rest binarization
y_test_bin = label_binarize(y_test, classes=rf.classes_)

# 2. micro-AUCPR
aucpr_micro = average_precision_score(
    y_test_bin,
    y_score,
    average="micro"
)

print(f"AUCPR: {aucpr_micro:.3f}")


AUCPR: 0.884


In [37]:
zero_std_ratio = (
    (df["std_bytes"] == 0)
    .groupby(df["category"])
    .mean()
    .sort_values(ascending=False)
)

print(zero_std_ratio)


category
remote_controller     0.960142
robot_cleaner         0.959718
door_lock             0.959141
environment_sensor    0.946152
doorphone             0.939803
light                 0.912490
network_camera        0.908486
plug                  0.850898
iot_gateway           0.841711
tv                    0.647172
smart_speaker         0.401493
Name: std_bytes, dtype: float64


In [38]:
device_usage = (
    df.groupby("device_id")["second"]
      .nunique()
      .sort_values(ascending=False)
)

print(device_usage)


device_id
planex_camera_one_shot!                  1466860
google_home_gen1                         1300197
sony_bravia                              1183909
jvc_kenwood_cu-hb1                       1110552
sony_smart_speaker                        853052
panasonic_doorphone                       839856
au_wireless_adapter                       632836
i-o_data_qwatch                           568097
philips_hue_bridge                        387144
amazon_echo_gen2                          309742
line_clova_wave                           287216
candy_house_sesami_wi-fi_access_point     255287
bitfinder_awair_breathe_easy              253842
powerelectric_wi-fi_plug                  239105
link_japan_eremote                        211310
planex_smacam_outdoor                     165634
qrio_hub                                  150674
sony_network_camera                       138889
au_network_camera                         135782
jvc_kenwood_hdtv_ip_camera                133240
irobot_roo